In [1]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KernelDensity

from batchdetect.loader import *
from batchdetect.mixture import HeavyMixture,parametric_bootstrap_lrt

In [2]:
with open('TST_Extracted_Features.p','rb') as f:
    myDict = pickle.load(f)

with open('PowerSocialData.p','rb') as f:
    myDict2 = pickle.load(f)

In [3]:
def null_factory() -> HeavyMixture:
    return HeavyMixture(
        n_components=1,
        component_distribution='gennorm',
        n_init=3,
        max_iter=1000,
    )

def alt_factory() -> HeavyMixture:
    return HeavyMixture(
        n_components=2,
        component_distribution='gennorm',
        n_init=3,
        max_iter=1000,
    )

In [4]:
def select_train_indices(y_mouse_tst, n_train, seed=42):
    """
    Select indices belonging to a random subset of n_train unique values in y_mouse_tst.

    Parameters
    ----------
    y_mouse_tst : array-like
        Input vector of labels or values.
    n_train : int
        Number of unique values to select.
    seed : int, optional
        Random seed for reproducibility.

    Returns
    -------
    mask : np.ndarray, dtype=bool
        Boolean vector of length len(y_mouse_tst) with True for selected indices.
    """
    y = np.asarray(y_mouse_tst)
    unique_vals = np.unique(y)

    if n_train > len(unique_vals):
        raise ValueError("n_train exceeds number of unique values in y_mouse_tst.")

    rng = np.random.default_rng(seed)
    selected_vals = rng.choice(unique_vals, size=n_train, replace=False)

    mask = np.isin(y, selected_vals)

    return mask


def filter_bottom_1pct(X, y, z, model_pca):
    """
    Compute PCA-based density scores and remove the bottom 1 percent.

    Parameters
    ----------
    X : array-like, shape (n_samples, n_features)
        Input matrix.
    y : array-like, shape (n_samples,)
        Associated vector of labels or values.
    z : array-like, shape (n_samples,)
        Second associated vector of labels or values.
    model_pca : object
        A fitted model with a score_samples method.

    Returns
    -------
    scores_filt : np.ndarray
        Filtered score_samples output.
    y_filt : np.ndarray
        y with bottom 1 percent removed.
    z_filt : np.ndarray
        z with bottom 1 percent removed.
    """
    X = np.asarray(X)
    y = np.asarray(y)
    z = np.asarray(z)

    # Compute PCA density scores
    scores_test = model_pca.score_samples(X)

    # Determine threshold for bottom 1 percent
    thresh = np.percentile(scores_test, .1)

    # Mask for keeping samples above the threshold
    mask = scores_test > thresh

    # Apply mask
    scores_filt = scores_test[mask]
    y_filt = y[mask]
    z_filt = z[mask]

    return scores_filt, y_filt, z_filt

def mean_scores_by_mouse_and_task(mouse_t_filt, scores_tst_filt, task_t_filt):
    """
    Returns an array of shape (n_unique_mice, 2), where:
      result[i, 0] = mean score for mouse i with task == 0
      result[i, 1] = mean score for mouse i with task == 1
    """
    mice = np.unique(mouse_t_filt)
    tasks = np.array([0, 1])

    # Masks for mouse and task membership
    mouse_mask = mouse_t_filt[:, None] == mice[None, :]      # (N, M)
    task_mask  = task_t_filt[:, None]  == tasks[None, :]     # (N, 2)

    # Combine into (N, M, 2)
    combined_mask = mouse_mask[:, :, None] & task_mask[:, None, :]

    # Weighted sums and counts
    num   = np.sum(scores_tst_filt[:, None, None] * combined_mask, axis=0)  # (M, 2)
    denom = np.sum(combined_mask, axis=0)                                   # (M, 2)

    return num / denom


In [5]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity

def _silverman_bandwidth(x):
    """Silverman's rule-of-thumb bandwidth for 1D data."""
    x = np.asarray(x)
    x = x[~np.isnan(x)]
    n = x.size
    if n < 2:
        return 1.0  # fallback
    std = np.std(x, ddof=1)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    sigma = min(std, iqr / 1.34) if iqr > 0 else std
    if sigma == 0:
        sigma = 1.0
    return 0.9 * sigma * n ** (-1.0 / 5.0)


def plot_kde_four(v1, v2, v3, v4, labels=None):
    """
    Plot KDEs for four 1D vectors v1..v4 using a compact-support kernel
    (Epanechnikov) with bandwidth inferred via Silverman's rule.

    Parameters
    ----------
    v1, v2, v3, v4 : array-like
        Input 1D vectors.
    labels : list of str or None
        Optional labels for legend (len(labels) == 4).
    """
    vectors = [np.asarray(v) for v in (v1, v2, v3, v4)]

    # Flatten and drop NaNs
    vectors = [v[np.isfinite(v)] for v in vectors]

    # Build a common grid over all data
    all_vals = np.concatenate(vectors)
    data_min, data_max = np.min(all_vals), np.max(all_vals)
    margin = 0.05 * (data_max - data_min if data_max > data_min else 1.0)
    xs = np.linspace(data_min - margin, data_max + margin, 400)[:, None]

    # Pastel color palette
    colors = [
        "#a6cee3",  # light blue
        "#b2df8a",  # light green
        "#fdbf6f",  # light orange
        "#cab2d6",  # light purple
    ]

    if labels is None:
        labels = ["v1", "v2", "v3", "v4"]

    # Create figure
    fig, ax = plt.subplots(figsize=(7, 4))

    for v, color, label in zip(vectors, colors, labels):
        if v.size == 0:
            continue

        # Epanechnikov kernel has compact support
        h = _silverman_bandwidth(v)
        kde = KernelDensity(kernel="gaussian", bandwidth=h)
        kde.fit(v[:, None])

        log_dens = kde.score_samples(xs)
        dens = np.exp(log_dens)

        ax.plot(xs[:, 0], dens, label=label, linewidth=2.2, alpha=0.9, color=color)

    # Styling: no "shitty" grid, just clean axes
    ax.grid(False)

    # Remove top and right spines, thicken bottom and left
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_linewidth(1.5)
    ax.spines["left"].set_linewidth(1.5)

    # Larger labels and title
    ax.set_xlabel("Likelihood", fontsize=20)
    ax.set_ylabel("Density", fontsize=20)
    ax.set_title('Electrophysiology Likelihood Distribution',fontsize=24)
    ax.tick_params(axis="both", which="major", labelsize=12)

    ax.legend(frameon=False, fontsize=12)
    
    fig.tight_layout()
    return fig, ax


In [6]:
y_task_tst = myDict['task']
y_mouse_tst = myDict['mouse']
y_mouse_soc = myDict2['y_mouse']
y_task_soc = myDict2['y_social']

In [7]:
X1 = myDict['power']
#XX1 = StandardScaler().fit_transform(X1)


In [8]:
X1p = np.log(X1[:10000])

In [9]:
X2p = np.log(X1[1*10000:2*10000])
X3p = np.log(X1[2*10000:3*10000])
X4p = np.log(X1[3*10000:4*10000])
X5p = np.log(X1[4*10000:5*10000])
X6p = np.log(X1[5*10000:])

In [10]:
X1l = np.vstack((X1p,X2p,X3p,X4p,X5p,X6p))

In [11]:
X2 = myDict2['X_psd']
Xllist = []
for i in range(13):
    Xllist.append(np.log(X2[i*10000:(i+1)*10000]))
Xllist.append(np.log(X2[130000:]))

In [12]:
X2l = np.vstack(Xllist)

In [13]:
XX1 = StandardScaler().fit_transform(X1l)
XX2 = StandardScaler().fit_transform(X2l)

In [14]:
power_list_tst = {}
idx = 2
power_list_tst['Amy'] = XX1[:,idx*56:(idx+1)*56]

power_list_tst['NAc'] = (XX1[:,0*56:(0+1)*56] + XX1[:,1*56:(1+1)*56])/2

idx = 4
power_list_tst['MD_Thal'] = XX1[:,idx*56:(idx+1)*56]

idx = 3
power_list_tst['IL'] = XX1[:,idx*56:(idx+1)*56]

idx = 5
power_list_tst['PrL'] = XX1[:,idx*56:(idx+1)*56]

idx = 6
power_list_tst['VTA'] = XX1[:,idx*56:(idx+1)*56]

idx = 7
power_list_tst['Hip'] = (XX1[:,idx*56:(idx+1)*56] + XX1[:,9*56:(9+1)*56])/2

In [15]:
power_list_social = {}
idx = 0
power_list_social['Amy'] = XX2[:,idx*56:(idx+1)*56]
idx = 1
power_list_social['Cg'] = XX2[:,idx*56:(idx+1)*56]
idx = 2
power_list_social['IL'] = XX2[:,idx*56:(idx+1)*56]
idx = 3
power_list_social['PrL'] = XX2[:,idx*56:(idx+1)*56]
idx = 4
power_list_social['NAc'] = XX2[:,idx*56:(idx+1)*56]
idx = 5
power_list_social['Hip'] = XX2[:,idx*56:(idx+1)*56]
idx = 6
power_list_social['MD_Thal'] = XX2[:,idx*56:(idx+1)*56]
idx = 7
power_list_social['VTA'] = XX2[:,idx*56:(idx+1)*56]

## Common are Amy, NAc, MD_Thal, IL, PrL, Hip, VTA

In [16]:
rd = power_list_social
X_social = np.hstack((rd['Amy'],rd['NAc'],rd['MD_Thal'],rd['IL'],rd['PrL'],rd['Hip'],rd['VTA']))
rd = power_list_tst
X_tst = np.hstack((rd['Amy'],rd['NAc'],rd['MD_Thal'],rd['IL'],rd['PrL'],rd['Hip'],rd['VTA']))

In [19]:
def analysis(X_social,y_mouse_soc,y_task_soc,X_tst,y_mouse_tst,y_task_tst,n_train,seed):
    idx_train = select_train_indices(y_mouse_soc, n_train, seed=42)
    results = {}
    model_pca = PCA(20)
    model_pca.fit(X_social[idx_train])
    results['model_pca'] = model_pca
    results['model_pca'] = idx_train
    
    scores_filt, mouse_s_filt, task_s_filt = filter_bottom_1pct(X_social[idx_train==False],
                                                 y_mouse_soc[idx_train==False],
                                                 y_task_soc[idx_train==False],
                                                 model_pca)
    scores_tst_filt, mouse_t_filt, task_t_filt = filter_bottom_1pct(X_tst,
                                                 y_mouse_tst,
                                                 y_task_tst,
                                                 model_pca)
    
    savg_tst = np.array([np.mean(scores_tst_filt[mouse_t_filt==m]) for m in np.unique(mouse_t_filt)])
    savg_soc = np.array([np.mean(scores_filt[mouse_s_filt==m]) for m in np.unique(mouse_s_filt)])
    savg_tot = np.concatenate((savg_tst,savg_soc))
    results['savg_tst_med'] = np.median(savg_tst)
    results['savg_soc_med'] = np.median(savg_soc)
    
    results['res_tst_avg'] = parametric_bootstrap_lrt(savg_tst,null_factory,alt_factory,1000,random_state=seed)
    results['res_soc_avg'] = parametric_bootstrap_lrt(savg_soc,null_factory,alt_factory,1000,random_state=seed)
    results['res_tot_avg'] = parametric_bootstrap_lrt(savg_tot,null_factory,alt_factory,1000,random_state=seed)

    savg_tst_mat = mean_scores_by_mouse_and_task(mouse_t_filt, scores_tst_filt, task_t_filt)
    savg_soc_mat = mean_scores_by_mouse_and_task(mouse_s_filt, scores_filt, task_s_filt)

    savg_tot_tst = np.concatenate((savg_tst_mat[:,0],savg_tst_mat[:,1]))
    savg_tot_soc = np.concatenate((savg_soc_mat[:,0],savg_soc_mat[:,1]))
    
    results['res_tst_sep'] = parametric_bootstrap_lrt(savg_tot_tst,null_factory,alt_factory,1000,random_state=seed)
    results['res_soc_sep'] = parametric_bootstrap_lrt(savg_tot_soc,null_factory,alt_factory,1000,random_state=seed)

    savg_tot_tot = np.concatenate((savg_tot_soc,savg_tot_tst))
    results['res_tot'] = parametric_bootstrap_lrt(savg_tot_tot,null_factory,alt_factory,1000,random_state=seed)
    return results

In [20]:
results = analysis(X_social,y_mouse_soc,y_task_soc,X_tst,y_mouse_tst,y_task_tst,15,2021)

In [29]:
results_list = []
for i in range(30):
    print(i)
    results_list.append(analysis(X_social,y_mouse_soc,y_task_soc,X_tst,y_mouse_tst,y_task_tst,15,2021+i))

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26


KeyboardInterrupt: 

In [32]:
def print_results(results):
    print('Overall separated',results['res_tot']['p_value'],results['res_tot']['p_value']<.05)
    print('TST separated',results['res_tst_sep']['p_value'],results['res_tst_sep']['p_value']<.05)
    print('SOC separated',results['res_soc_sep']['p_value'],results['res_soc_sep']['p_value']<.05)
    print('Overall avg',results['res_tot_avg']['p_value'],results['res_tot_avg']['p_value']<.05)
    print('TST avg',results['res_tst_avg']['p_value'],results['res_tst_avg']['p_value']<.05)
    print('Soc avg',results['res_soc_avg']['p_value'],results['res_soc_avg']['p_value']<.05)

In [28]:
print_results(results)

Overall separated 0.001998001998001998 True
TST separated 0.6973026973026973 False
SOC separated 0.04095904095904096 True
Overall avg 0.013986013986013986 False
TST avg 0.12387612387612387 False
Soc avg 0.015984015984015984 True


In [36]:
results_list2 = []
for i in range(40):
    print(i)
    results_list2.append(analysis(X_social,y_mouse_soc,y_task_soc,X_tst,y_mouse_tst,y_task_tst,15,1993-i))
    with open('Ephys_2.p','wb') as f:
        pickle.dump({'res':results_list2},f)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39


In [33]:
for i in range(10):
    print('----------------------------------------------')
    print(i)
    print_results(results_list[i])

----------------------------------------------
0
Overall separated 0.002997002997002997 True
TST separated 0.6953046953046953 False
SOC separated 0.03496503496503497 True
Overall avg 0.005994005994005994 True
TST avg 0.13786213786213786 False
Soc avg 0.017982017982017984 True
----------------------------------------------
1
Overall separated 0.004995004995004995 True
TST separated 0.6533466533466533 False
SOC separated 0.054945054945054944 False
Overall avg 0.007992007992007992 True
TST avg 0.13786213786213786 False
Soc avg 0.016983016983016984 True
----------------------------------------------
2
Overall separated 0.002997002997002997 True
TST separated 0.6823176823176823 False
SOC separated 0.04595404595404595 True
Overall avg 0.015984015984015984 True
TST avg 0.12187812187812187 False
Soc avg 0.012987012987012988 True
----------------------------------------------
3
Overall separated 0.002997002997002997 True
TST separated 0.6503496503496503 False
SOC separated 0.04295704295704296 T

In [38]:
len(results_list)

26

In [34]:
with open('Ephys_1.p','wb') as f:
    pickle.dump({'res':results_list},f)

In [39]:
results_list3 = []
for i in range(34):
    print(i)
    results_list2.append(analysis(X_social,y_mouse_soc,y_task_soc,X_tst,y_mouse_tst,y_task_tst,15,1942-i))
    with open('Ephys_2.p','wb') as f:
        pickle.dump({'res':results_list2},f)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33


In [40]:
results_combined = results_list + results_list2 + results_list3

In [1]:
#with open('Ephys_soc.p','wb') as f:
#    pickle.dump({'res':results_combined},f)

NameError: name 'pickle' is not defined

In [44]:
n_perm = len(results_combined)
rej_tot_sep = np.zeros(n_perm)
rej_tot_avg = np.zeros(n_perm)
pval_tot_sep = np.zeros(n_perm)
pval_tot_avg = np.zeros(n_perm)

rej_tst_sep = np.zeros(n_perm)
rej_tst_avg = np.zeros(n_perm)
pval_tst_sep = np.zeros(n_perm)
pval_tst_avg = np.zeros(n_perm)

rej_soc_sep = np.zeros(n_perm)
rej_soc_avg = np.zeros(n_perm)
pval_soc_sep = np.zeros(n_perm)
pval_soc_avg = np.zeros(n_perm)

for i in range(n_perm):
    rej_tot_sep[i] = results_combined[i]['res_tot']['p_value']<.05
    rej_tot_avg[i] = results_combined[i]['res_tot_avg']['p_value']<.05
    pval_tot_sep[i] = results_combined[i]['res_tot']['p_value']
    pval_tot_avg[i] = results_combined[i]['res_tot_avg']['p_value']
    
    rej_tst_sep[i] = results_combined[i]['res_tst_sep']['p_value']<.05
    rej_tst_avg[i] = results_combined[i]['res_tst_avg']['p_value']<.05
    pval_tst_sep[i] = results_combined[i]['res_tst_sep']['p_value']
    pval_tst_avg[i] = results_combined[i]['res_tst_avg']['p_value']
    
    rej_soc_sep[i] = results_combined[i]['res_soc_sep']['p_value']<.05
    rej_soc_avg[i] = results_combined[i]['res_soc_avg']['p_value']<.05
    pval_soc_sep[i] = results_combined[i]['res_soc_sep']['p_value']
    pval_soc_avg[i] = results_combined[i]['res_soc_avg']['p_value']



In [45]:
print('Rejection rate total separated: ',np.mean(rej_tot_sep))
print('Rejection rate total averaged: ',np.mean(rej_tot_avg))

print('Rejection rate tst separated: ',np.mean(rej_tst_sep))
print('Rejection rate tst averaged: ',np.mean(rej_tst_avg))

print('Rejection rate soc separated: ',np.mean(rej_soc_sep))
print('Rejection rate soc averaged: ',np.mean(rej_soc_avg))

Rejection rate total separated:  1.0
Rejection rate total averaged:  1.0
Rejection rate tst separated:  0.0
Rejection rate tst averaged:  0.0
Rejection rate soc separated:  0.63
Rejection rate soc averaged:  1.0


In [47]:
print('Average p-value total separated: ',np.mean(pval_tot_sep))
print('Average p-value total averaged: ',np.mean(pval_tot_avg))

print('Average p-value tst separated: ',np.mean(pval_tst_sep))
print('Average p-value tst averaged: ',np.mean(pval_tst_avg))

print('Average p-value soc separated: ',np.mean(pval_soc_sep))
print('Average p-value soc averaged: ',np.mean(pval_soc_avg))

Average p-value total separated:  0.0045554445554445554
Average p-value total averaged:  0.01271728271728272
Average p-value tst separated:  0.6783516483516482
Average p-value tst averaged:  0.12283716283716284
Average p-value soc separated:  0.0486013986013986
Average p-value soc averaged:  0.014215784215784216
